# Building a CNN Feature Block with Conv2d and MaxPool2d

**Learning goals:**

- Explain the distinct roles of Conv2d (feature extraction) and MaxPool2d (dimension reduction) on image data.
- Configure PyTorch nn.Conv2d and nn.MaxPool2d parameters.
- Apply a Conv-Pool block to a handwritten digit image.

**Prerequisites:**

- Basic Python functions and classes, image pixels and tensors from Sessions 9–12.
- Python with PyTorch installed; a CPU is sufficient.
- Open the notebook with the AI_SS course folder as the working directory. The digit data is `sample_data/mnist_train_small.csv` (label first, followed by 784 pixels; no header).
- In Google Colab, upload the course CSV into a `sample_data` folder in the runtime working directory. Colab normally includes the required libraries.
- Internet access for the optional browser visualiser; the Python activities use the supplied CSV without downloads.


## 0. Review: CNN Network Structure

![CNN feature extraction from dog image](assets/cnn-dog-feature-extraction.png)

**How does a neural network turn raw image pixels into meaningful visual features?**



A CNN learns features such as edges, textures and shapes from image pixels, then uses them to predict a class.

![Typical CNN architecture showing an input image, convolution and pooling stages, fully connected layers and an output layer.](assets/typical-cnn-architecture.png)

- **Convolution layer:** Extracts features such as edges and textures.
- **Max-pooling layer:** Reduces feature-map size by keeping the strongest local responses.
- **Fully connected layer:** Combines the features to predict a class.

**Interactive 3D demo:** [Adam Harley - CNN 3D Visualisation](https://adamharley.com/nn_vis/cnn/3d.html). Draw a handwritten digit to see how a CNN processes and recognises it.

## 1. PyTorch convolution and max-pooling layers

| CNN layer | PyTorch layer | Main role |
|---|---|---|
| Convolution layer | `torch.nn.Conv2d` | Extracts features using trainable filters. |
| Max-pooling layer | `torch.nn.MaxPool2d` | Reduces feature-map height and width by selecting local maxima. |

Both work on the spatial dimensions of an image or feature map.


**What is a tensor?**

A tensor stores numerical data. It can represent a single number (zero dimensions), a vector (one dimension), a matrix (two dimensions) or an array with more dimensions. PyTorch uses tensors to store image pixels, feature maps and model outputs.

A tensor's **shape** gives the size of each dimension. For image inputs, PyTorch commonly uses `[N, C, H, W]`:

- **`N`:** Number of images processed together. Use `1` for a single image.
- **`C`:** Number of channels: `1` for greyscale or `3` for RGB images.
- **`H`:** Image height in pixels.
- **`W`:** Image width in pixels.

For example, `[1, 1, 28, 28]` represents one 28 × 28 greyscale image. `[1, 16, 28, 28]` represents 16 feature maps for that image, each 28 × 28. A feature map is a channel of responses rather than a colour channel.

The examples below use floating-point tensors. The layers change their values and may change their shape.


### 1.1 Conv2d: extract features

**Conv2d** applies trainable filters to extract features from an image.

**Usage example:**

```python
nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1)
```

This creates a convolution layer. When an image tensor passes through this layer:

- **Input:** `[1, 1, 28, 28]` — one image, one greyscale channel, 28 pixels high and 28 pixels wide.
- **Output:** `[1, 16, 28, 28]` — one image represented by 16 feature maps, each 28 × 28. Padding keeps the spatial size unchanged for these settings.

**Parameters:**

- **`in_channels=1`:** Accept one input channel for greyscale images.
- **`out_channels=16`:** Use 16 trainable filters to produce 16 feature maps.
- **`kernel_size=3`:** Use a 3 × 3 filter.
- **`stride=1`:** Move the filter one pixel at each step.
- **`padding=1`:** Add a one-pixel border of zeros on each side.

**Input and output sizes:**

The number of images stays the same. The input channel count must match `in_channels`; `out_channels` sets the number of output feature maps.

With the default `dilation=1`, use `kernel_size`, `stride` and `padding` to calculate the spatial dimensions:

$$
\text{output height} = \left\lfloor \frac{\text{input height} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} + 1 \right\rfloor
$$

$$
\text{output width} = \left\lfloor \frac{\text{input width} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} + 1 \right\rfloor
$$

The floor symbols $\lfloor\cdot\rfloor$ mean round down to the nearest integer. For tuple settings, use the corresponding `kernel_size`, `stride` and `padding` value for each dimension.

For the example above, input height and width are both 28, `kernel_size=3`, `stride=1` and `padding=1`:

$$
\text{output height} = \text{output width} = \left\lfloor \frac{28 + 2\times1 - 3}{1} + 1 \right\rfloor = 28
$$

The output is therefore `[1, 16, 28, 28]`. With `padding=0` instead, each feature map would be 26 × 26.

[Official documentation: Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)


### 1.2 MaxPool2d: reduce spatial dimensions

**MaxPool2d** keeps the largest value in each local window. It works on each channel separately and has no trainable weights.

**Usage example:**

```python
nn.MaxPool2d(kernel_size=2, stride=2)
```

This creates a max-pooling layer. When the convolution's feature maps pass through this layer:

- **Input:** `[1, 16, 28, 28]` — one image represented by 16 feature maps, each 28 × 28.
- **Output:** `[1, 16, 14, 14]` — the same 16 feature maps, each reduced to 14 × 14. The batch size and channel count stay unchanged.

**Parameters:**

- **`kernel_size=2`:** Select the largest value in each 2 × 2 window.
- **`stride=2`:** Move the window two pixels at each step, halving the even height and width in this example.

Padding is omitted, so its default value is `0`.

**Input and output sizes:**

MaxPool2d keeps the number of images and channels unchanged. With the defaults `dilation=1` and `ceil_mode=False`, calculate height and width as:

$$
\text{output height} = \left\lfloor \frac{\text{input height} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} + 1 \right\rfloor
$$

$$
\text{output width} = \left\lfloor \frac{\text{input width} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} + 1 \right\rfloor
$$

Round down to the nearest integer. For tuple settings, use the corresponding values for height and width.

For this example, input height and width are both 28, `kernel_size=2`, `stride=2` and `padding=0`:

$$
\text{output height} = \text{output width} = \left\lfloor \frac{28 + 2\times0 - 2}{2} + 1 \right\rfloor = 14
$$

The shape changes from `[1, 16, 28, 28]` to `[1, 16, 14, 14]`. The 16 channels remain; only height and width are reduced.

Together, **Conv2d extracts features** and **MaxPool2d reduces their spatial size**. ReLU is commonly placed between them to add non-linearity without changing shape.

[Official documentation: MaxPool2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)


## 2. Live Coding

Inherit from `nn.Module` and define the layers in `__init__`. The forward pass is **Conv2d → ReLU → MaxPool2d → Flatten → Linear**.

The feature block produces 16 maps of 14 × 14. Flatten converts them into 3,136 values, and a single Linear layer returns ten class scores (logits).


In [ ]:
import torch
from torch import nn

class DigitCNN(nn.Module):
    """Extract image features and produce ten digit-class scores."""

    def __init__(self):
        super().__init__()  # Initialise the parent module before adding layers.
        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,  # Accept one greyscale channel.
                out_channels=16,  # Produce 16 feature maps.
                kernel_size=3,  # Apply a 3 × 3 filter.
                stride=1,  # Move one pixel at each step.
                padding=1,  # Keep the 28 × 28 spatial size.
            ),
            nn.ReLU(),  # Replace negative responses with zero.
            nn.MaxPool2d(kernel_size=2, stride=2),  # Reduce each map to 14 × 14.
        )
        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),  # Flatten the maps while keeping the batch axis.
            nn.Linear(in_features=16 * 14 * 14, out_features=10),  # Produce ten class scores directly from the flattened features.
        )

    def forward(self, images):
        features = self.features(images)  # Extract and pool local image features.
        return self.classifier(features)  # Convert the features into ten class scores.

torch.manual_seed(42)  # Reproduce the initial random model weights.
model = DigitCNN()  # Create the same feature block and classifier as the lesson.
print(model)
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))
print('Weights are randomly initialised; this model has not been trained.')

**Run the model**

Create the model, switch to evaluation mode and run a forward pass. The output contains ten scores; `argmax` selects the highest-scoring class. The model is untrained, so this only illustrates the steps.


In [ ]:
# Run inference on a single exampleimage.
images = torch.zeros(1, 1, 28, 28)  # Batch size, channels, height and width.

model.eval()  # Select evaluation behaviour.

with torch.no_grad():  # Run inference without recording gradients.
    logits = model(images)  # Return ten class scores for the input image.
    predictions = logits.argmax(dim=1)  # Select the highest-scoring digit class.

print(logits.shape)  # Expected: torch.Size([1, 10]).
print(predictions)  # One predicted class for the input image.

## 3. Hands-on Notebook

[Download the practice notebook](<Session12.1 CNN Practice.ipynb>){download="Session12.1 CNN Practice.ipynb"}

Use Google Colab to run the practice notebook:

1. Download the practice notebook using the link above.
2. Open [Google Colab](https://colab.research.google.com/) and sign in with your Google account.
3. Choose **File → Upload notebook** and select `Session12.1 CNN Practice.ipynb`.
4. Click **Connect** to start a runtime. A CPU is sufficient for this activity.
5. In the **Files** sidebar, check for `sample_data/mnist_train_small.csv`. If it is missing, [download the digit CSV](sample_data/mnist_train_small.csv){download="mnist_train_small.csv"}, create a `sample_data` folder if needed and upload the CSV into it. Keep the notebook's `DATA_PATH` unchanged.
6. Run the cells from top to bottom. Inspect the digit, image batch and outputs from Conv2d, ReLU and MaxPool2d. Explain why pooling changes 28 × 28 maps into 14 × 14 maps while keeping 16 channels.

The practice notebook also shows Flatten and Linear layers. The model is untrained, so its predicted classes do not demonstrate recognition accuracy.

[Colab help](https://research.google.com/colaboratory/faq.html)


## 4. Check for Understanding

**Interactive Question:**

- How many feature maps did your block create, and what is their final size?

**Concept Verification:**

- Explain the purpose of Conv2d and MaxPool2d.
- Why did we set `in_channels=1` and `out_channels=16`?
